In [ ]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import sklearn
import string
import re
from IPython.display import display, Latex, Markdown

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt')

lemmatizer=nltk.stem.wordnet.WordNetLemmatizer()
stopwords=nltk.corpus.stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/Mini_Research_Project"
!pwd

/content/drive/MyDrive/Mini_Research_Project
/content/drive/MyDrive/Mini_Research_Project


In [ ]:
def Load_Data(file_path, sheet_name, tweet_column, class_column, ignore_class, screen_name):
    data = pd.read_excel(file_path, sheet_name=sheet_name, header=1, usecols=f"{tweet_column}:{class_column}")
    data.columns = ["Tweet", "Label"]
    data = data[data['Label'].isin([-1, 0, 1])]
    data.loc[:, 'Tweet'] = data['Tweet'].fillna("UNKNOWN")

    data.loc[:, 'Screen_Name'] = screen_name

    return data[['Screen_Name', 'Tweet', 'Label']].values.tolist()

file_path = "/content/drive/MyDrive/Mini_Research_Project/training-Obama-Romney-tweets.xlsx"
obama_sheet_name, romney_sheet_name = "Obama", "Romney"
tweet_column_index, class_column_index = "D", "E"
ignore_class_label = 2

obama_data = Load_Data(file_path, obama_sheet_name, tweet_column_index, class_column_index, ignore_class_label, screen_name='Obama')

romney_data = Load_Data(file_path, romney_sheet_name, tweet_column_index, class_column_index, ignore_class_label, screen_name='Romney')

combined_data = pd.DataFrame(obama_data + romney_data, columns=['Screen_Name', 'Tweet', 'Label'])

print("Rows with Screen_Name 'Obama':")
print(combined_data[combined_data['Screen_Name'] == 'Obama'].head())

print("\nRows with Screen_Name 'Romney':")
print(combined_data[combined_data['Screen_Name'] == 'Romney'].head())



Rows with Screen_Name 'Obama':
  Screen_Name                                              Tweet  Label
0       Obama  Kirkpatrick, who wore a baseball cap embroider...      0
1       Obama  #<e>obama</e> debates that Cracker Ass Cracker...      1
2       Obama  @Hollivan @hereistheanswer  Youre missing the ...      0
3       Obama  I was raised as a Democrat  left the party yea...     -1
4       Obama  The <e>Obama camp</e> can't afford to lower ex...      0

Rows with Screen_Name 'Romney':
     Screen_Name                                              Tweet  Label
5471      Romney  Insidious!<e>Mitt Romney</e>'s Bain Helped Phi...     -1
5472      Romney  .@WardBrenda @shortwave8669 @allanbourdius you...     -1
5473      Romney  <e>Mitt Romney</e> still doesn't <a>believe</a...     -1
5474      Romney  <e>Romney</e>'s <a>tax plan</a> deserves a 2nd...     -1
5475      Romney  Hope <e>Romney</e> debate prepped w/ the same ...      1


In [ ]:
posMapping = {
    "N":'n',
    "V":'v',
    "J":'a',
    "R":'r'
}
def process(text, lemmatizer=nltk.stem.wordnet.WordNetLemmatizer()):
    """ Normalizes case and handles punctuation
    Inputs:
        text: str: raw text
        lemmatizer: an instance of a class implementing the lemmatize() method
                    (the default argument is of type nltk.stem.wordnet.WordNetLemmatizer)
    Outputs:
        list(str): tokenized text
    """
    text = text.replace("'s", "").replace("'", "").replace("-", " ").lower()
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    tokens = re.findall(r"\b\w+\b|\d+:\d+[ap]m|…", text)
    return [lemmatizer.lemmatize(token, posMapping.get(pos[0].upper(), 'n')) for token, pos in nltk.pos_tag(tokens)]

In [ ]:
def process_all(df, lemmatizer=nltk.stem.wordnet.WordNetLemmatizer()):
    process_t = df['Tweet'].apply(lambda text: process(text, lemmatizer))
    df['Tweet'] = process_t
    return df

In [ ]:
processed_tweets = process_all(combined_data)

print("Rows with Screen_Name 'Obama':")
print(processed_tweets[combined_data['Screen_Name'] == 'Obama'].head())


print("\nRows with Screen_Name 'Romney':")
print(processed_tweets[combined_data['Screen_Name'] == 'Romney'].head())

Rows with Screen_Name 'Obama':
  Screen_Name                                              Tweet  Label
0       Obama  [kirkpatrick, who, wear, a, baseball, cap, emb...      0
1       Obama  [e, obama, e, debate, that, cracker, as, crack...      1
2       Obama  [hollivan, hereistheanswer, youre, miss, the, ...      0
3       Obama  [i, be, raise, a, a, democrat, leave, the, par...     -1
4       Obama  [the, e, obama, camp, e, cant, afford, to, low...      0

Rows with Screen_Name 'Romney':
     Screen_Name                                              Tweet  Label
5471      Romney  [insidious, e, mitt, romney, e, bain, help, ph...     -1
5472      Romney  [wardbrenda, shortwave8669, allanbourdius, you...     -1
5473      Romney  [e, mitt, romney, e, still, doesnt, a, believe...     -1
5474      Romney  [e, romney, e, a, tax, plan, a, deserve, a, 2n...     -1
5475      Romney  [hope, e, romney, e, debate, prepped, w, the, ...      1


In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline


num_runs = 2

for run in range(num_runs):
    X_train, X_test, y_train, y_test = train_test_split(processed_tweets['Tweet'], processed_tweets['Label'], test_size=0.2, random_state=run*10)
    X_train_text = X_train.apply(lambda x: ' '.join(x))
    X_test_text = X_test.apply(lambda x: ' '.join(x))
    pipeline_svc = Pipeline([
        ('vectorizer', TfidfVectorizer(max_features=1000, sublinear_tf=True)),
        ('classifier', SVC(random_state=42))
    ])
    svc_params = {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf']
    }

    random_search_svc = RandomizedSearchCV(pipeline_svc, svc_params, scoring='accuracy', cv=3, n_jobs=-1, n_iter=6, random_state=run*10)
    random_search_svc.fit(X_train_text, y_train)
    best_params_svc_random = random_search_svc.best_params_
    print(f"\nBest Parameters for SVC (RandomizedSearchCV - Run {run + 1}):", best_params_svc_random)
    final_svc_model_random = random_search_svc.best_estimator_
    y_pred_svc_random = final_svc_model_random.predict(X_test_text)
    accuracy_svc_random = accuracy_score(y_test, y_pred_svc_random)
    report_svc_random = classification_report(y_test, y_pred_svc_random)

    print(f"\nSVC Model Results (RandomizedSearchCV - Run {run + 1}):")
    print("Accuracy:", accuracy_svc_random)
    print("\nClassification Report:")
    print(report_svc_random)



Best Parameters for SVC (RandomizedSearchCV - Run 1): {'classifier__kernel': 'rbf', 'classifier__C': 1}

SVC Model Results (RandomizedSearchCV - Run 1):
Accuracy: 0.5926258992805755

Classification Report:
              precision    recall  f1-score   support

          -1       0.58      0.79      0.67       933
           0       0.56      0.42      0.48       721
           1       0.67      0.49      0.56       570

    accuracy                           0.59      2224
   macro avg       0.60      0.57      0.57      2224
weighted avg       0.60      0.59      0.58      2224


Best Parameters for SVC (RandomizedSearchCV - Run 2): {'classifier__kernel': 'rbf', 'classifier__C': 1}

SVC Model Results (RandomizedSearchCV - Run 2):
Accuracy: 0.5962230215827338

Classification Report:
              precision    recall  f1-score   support

          -1       0.59      0.83      0.69       927
           0       0.59      0.40      0.48       745
           1       0.63      0.48      0.5

In [ ]:
test_data_obama = pd.read_excel("/content/drive/MyDrive/Mini_Research_Project/final-testData-no-label-Obama-tweets.xlsx", sheet_name="Obama", header = None,  usecols="B")
test_data_obama.rename(columns={ 1:'Tweet'}, inplace =True)
test_data_obama.head()

,Tweet
0,<e>Obama</e> has to maintain his professionali...
1,<e>Obama</e> went into the debate swinging and...
2,Ditto. I started @247LS 4 years ago. RT @bmorr...
3,I absolutely love <e>Obama</e>'s view in <a>im...
4,I'm agreeing completely with <e>Obama</e>'s st...


In [ ]:
test_data_romney = pd.read_excel("/content/drive/MyDrive/Mini_Research_Project/final-testData-no-label-Romney-tweets.xlsx", sheet_name="Romney", header = None,  usecols="B")
test_data_romney.rename(columns={ 1:'Tweet'}, inplace =True)
test_data_romney.head()

,Tweet
0,<e>Romney</e> got 3 less minutes and had to de...
1,<e>Mitt </e>is beating him UP! on his record...
2,I actually like <e>Romney </e>'s response to ...
3,Just for that <a>immigration statement </a>tha...
4,This man <e>Romney </e>is tearing this dude ...


In [ ]:
processed_obama_test_tweets = process_all(test_data_obama)
print(processed_obama_test_tweets.head())

                                               Tweet
0  [e, obama, e, have, to, maintain, his, profess...
1  [e, obama, e, go, into, the, debate, swinging,...
2  [ditto, i, start, 247ls, 4, year, ago, rt, bmo...
3  [i, absolutely, love, e, obama, e, view, in, a...
4  [im, agree, completely, with, e, obama, e, sta...


In [ ]:
processed_romney_test_tweets = process_all(test_data_romney)
print(processed_romney_test_tweets.head())

                                               Tweet
0  [e, romney, e, get, 3, less, minute, and, have...
1  [e, mitt, e, be, beat, him, up, on, his, recor...
2  [i, actually, like, e, romney, e, response, to...
3  [just, for, that, a, immigration, statement, a...
4  [this, man, e, romney, e, be, tear, this, dude...


In [ ]:
def save_predictions(data, model, output_filepath):
    text_data = data.apply(lambda x: ' '.join(x))
    preds_array = np.array(model.predict(text_data))
    with open(output_filepath, "w") as f:
        f.write("(setf x '(\n")
        for i in range(len(preds_array)):
            f.write(f"({i+1} {preds_array[i]})\n")
        f.write("))\n")

In [ ]:
output_directory = "/content/drive/MyDrive/Mini_Research_Project/"

save_predictions(processed_obama_test_tweets['Tweet'], final_svc_model_random, output_directory + "Obama.txt")
save_predictions(processed_romney_test_tweets['Tweet'], final_svc_model_random, output_directory + "Romney.txt")